## Step 1: Setup and Configuration

In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 23.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
     

In [3]:
import os
import sys
from pathlib import Path

# Set paths
SCRIPT_DIR = Path.cwd()
LLAMA_FACTORY_DIR = SCRIPT_DIR.parent
os.chdir(LLAMA_FACTORY_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"LLaMA-Factory directory: {LLAMA_FACTORY_DIR}")
try:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    print(f"✓ Unsloth imported successfully")
    print(f"  Version: {unsloth.__version__}")
except ImportError as e:
    print("✗ Unsloth not found!")
    print("  Install with: pip install \"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\"")
    raise e
try:
    from trl import GRPOConfig, GRPOTrainer
    print("✓ TRL imported successfully (GRPO support)")
except ImportError as e:
    print("✗ TRL not found!")
    print("  Install with: pip install trl")
    raise e

print("\nAll dependencies ready for Unsloth GRPO training!")

Working directory: /
LLaMA-Factory directory: /
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth imported successfully
  Version: 2025.11.2
✓ TRL imported successfully (GRPO support)

✅ All dependencies ready for Unsloth GRPO training!


## Step 2: Configure Model and Training Parameters

In [ ]:
# Model configuration
MODEL_NAME_OR_PATH = "content/drive/MyDrive/llama3_2_lora_sft_concat2"
MODEL_MAX_LENGTH = 5120

# Training configuration
OUTPUT_DIR = "saves/llama3-3b/grpo_unsloth_3modes"
EXPERIMENT_NAME = "grpo_unsloth_3modes"

# Dataset
DATA_DIR = LLAMA_FACTORY_DIR / "data"
GRPO_LOW_PATH = DATA_DIR / "combined_grpo_train_low.jsonl"
GRPO_MEDIUM_PATH = DATA_DIR / "combined_grpo_train_medium.jsonl"
GRPO_HIGH_PATH = DATA_DIR / "combined_grpo_train_high.jsonl"
VAL_PATH = DATA_DIR / "combined_val.jsonl"

# Performance settings
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 5.0e-7
MAX_STEPS = -1

# LoRA settings
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# GRPO
NUM_GENERATIONS = 4
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 4096
REASONING_TARGETS = {
    "low": {"min_tokens": 0, "max_tokens": 512, "weight": 1.0},
    "medium": {"min_tokens": 0, "max_tokens": 2048, "weight": 1.0},
    "high": {"min_tokens": 0, "max_tokens": 4096, "weight": 1.0}
}

print("="*70)
print("Configuration Summary")
print("="*70)
print(f"Model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"Output: {OUTPUT_DIR}")
print(f"\nDatasets (3 modes):")
print(f"  • Low:    {GRPO_LOW_PATH.name}")
print(f"  • Medium: {GRPO_MEDIUM_PATH.name}")
print(f"  • High:   {GRPO_HIGH_PATH.name}")
print(f"\nTraining:")
print(f"  • Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  • Gradient accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Effective batch: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Generations/prompt: {NUM_GENERATIONS}")
print("="*70)

Configuration Summary
Model: llama3_2_lora_sft_concat2
Output: saves/llama3-3b/grpo_unsloth_3modes

Datasets (3 modes):
  • Low:    combined_grpo_train_low.jsonl
  • Medium: combined_grpo_train_medium.jsonl
  • High:   combined_grpo_train_high.jsonl

Training:
  • Batch size: 1
  • Gradient accum: 8
  • Effective batch: 8
  • Learning rate: 5e-07
  • Generations/prompt: 4


## Step 3: Load and Combine 3 GRPO Datasets

In [8]:
import json
from datasets import Dataset, concatenate_datasets

def load_jsonl_with_mode(filepath, mode_name):
    """Load JSONL and add mode information."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if 'mode' not in item:
                item['mode'] = mode_name
            data.append(item)
    return data

print("Loading 3 GRPO datasets...")
print("="*60)

low_data = load_jsonl_with_mode(GRPO_LOW_PATH, "low")
print(f"✓ Low mode:    {len(low_data):,} examples")

medium_data = load_jsonl_with_mode(GRPO_MEDIUM_PATH, "medium")
print(f"✓ Medium mode: {len(medium_data):,} examples")

high_data = load_jsonl_with_mode(GRPO_HIGH_PATH, "high")
print(f"✓ High mode:   {len(high_data):,} examples")

combined_data = low_data + medium_data + high_data
print(f"\n✓ Combined:    {len(combined_data):,} total examples")

train_dataset = Dataset.from_list(combined_data)
print(f"\n✓ Created HuggingFace Dataset with {len(train_dataset)} examples")
print(f"  Columns: {train_dataset.column_names}")

val_data = load_jsonl_with_mode(VAL_PATH, "mixed")
eval_dataset = Dataset.from_list(val_data)
print(f"\n✓ Validation:  {len(eval_dataset):,} examples")

print("\n" + "="*60)
print("Dataset Distribution:")
print("="*60)
mode_counts = {"low": len(low_data), "medium": len(medium_data), "high": len(high_data)}
for mode, count in mode_counts.items():
    pct = 100 * count / len(combined_data)
    print(f"  {mode:8s}: {count:6,} ({pct:5.1f}%)")
print("="*60)

Loading 3 GRPO datasets...
✓ Low mode:    8,630 examples
✓ Medium mode: 9,463 examples
✓ High mode:   9,351 examples

✓ Combined:    27,444 total examples

✓ Created HuggingFace Dataset with 27444 examples
  Columns: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'combined_index']

✓ Validation:  1,742 examples

Dataset Distribution:
  low     :  8,630 ( 31.4%)
  medium  :  9,463 ( 34.5%)
  high    :  9,351 ( 34.1%)


In [ ]:
from pathlib import Path
import json

data_dir = Path("/LLaMA-Factory/data")
grpo_files = [
    "combined_grpo_train_low.jsonl",
    "combined_grpo_train_medium.jsonl",
    "combined_grpo_train_high.jsonl",
    "combined_val.jsonl"
]

print("Checking data files:")
print("="*60)

for filename in grpo_files:
    filepath = data_dir / filename
    if filepath.exists():
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = sum(1 for _ in f)
        with open(filepath, 'r', encoding='utf-8') as f:
            first_line = json.loads(f.readline())

        print(f"✓ {filename}")
        print(f"  Lines: {lines:,}")
        print(f"  Keys: {list(first_line.keys())}")
        if 'mode' in first_line:
            print(f"  Mode: {first_line['mode']}")
        if 'reasoning_tokens' in first_line:
            print(f"  Sample reasoning tokens: {first_line['reasoning_tokens']}")
    else:
        print(f"✗ {filename} - NOT FOUND")
    print()

Checking data files:
✓ combined_grpo_train_low.jsonl
  Lines: 8,630
  Keys: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'combined_index']
  Mode: low
  Sample reasoning tokens: 10.0

✓ combined_grpo_train_medium.jsonl
  Lines: 9,463
  Keys: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'combined_index']
  Mode: medium
  Sample reasoning tokens: 36.0

✓ combined_grpo_train_high.jsonl
  Lines: 9,351
  Keys: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'combined_index']
  Mode: high
  Sample reasoning tokens: 43.0

✓ combined_val.jsonl
  Lines: 1,742
  Keys: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'combined_index']
  Mode: low
  Sample reasoning tokens: 6.0



## Step 4: Load Model with Unsloth + Add LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

print("Loading model with Unsloth...")
print("="*60)

# Load SFT model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME_OR_PATH,
    max_seq_length=MODEL_MAX_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

print(f"✓ Loaded model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"✓ Max sequence length: {MODEL_MAX_LENGTH}")
print(f"✓ Device: {next(model.parameters()).device}")
print(f"✓ Dtype: {next(model.parameters()).dtype}")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print(f"\n✓ Added LoRA adapters:")
print(f"  • Rank: {LORA_RANK}")
print(f"  • Alpha: {LORA_ALPHA}")
print(f"  • Dropout: {LORA_DROPOUT}")
print(f"  • Gradient checkpointing: unsloth (optimized)")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*60)

Loading model with Unsloth...
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

content/drive/MyDrive/llama3_2_lora_sft_concat2 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
✓ Loaded model: llama3_2_lora_sft_concat2
✓ Max sequence length: 2048
✓ Device: cuda:0
✓ Dtype: torch.float16


Unsloth 2025.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✓ Added LoRA adapters:
  • Rank: 16
  • Alpha: 32
  • Dropout: 0.0
  • Gradient checkpointing: unsloth (optimized)

Trainable parameters: 24,313,856 / 3,237,063,680 (0.75%)


## Step 5: Create Multi-Mode Reward Function for 3 Reasoning Modes

In [ ]:
import re
import torch
from typing import List

class MultiModeRewardFunction:
    """Reward function for 3 reasoning modes: low, medium, high"""

    def __init__(self, mode_targets):
        self.mode_targets = mode_targets
        self.__name__ = "MultiModeRewardFunction"

    def count_reasoning_tokens(self, text):
        """Count tokens in <think> tags."""
        matches = re.findall(r"<think>(.*?)</think>", text, re.DOTALL)
        if not matches:
            return 0
        reasoning_text = " ".join(matches)
        return len(reasoning_text.split())

    def extract_answer(self, text):
        """Extract answer from \\boxed{}."""
        matches = re.findall(r"\\boxed\{([^}]+)\}", text)
        return matches[-1] if matches else ""

    def compute_mode_reward(self, text, target_mode):
        """Reward based on reasoning token count matching target mode."""
        tokens = self.count_reasoning_tokens(text)
        config = self.mode_targets[target_mode]

        min_tok = config["min_tokens"]
        max_tok = config["max_tokens"]

        if min_tok <= tokens <= max_tok:
            return 1.0
        elif tokens < min_tok:
            deficit = min_tok - tokens
            penalty = min(deficit / max(min_tok, 1), 1.0)
            return 1.0 - penalty
        else:
            excess = tokens - max_tok
            penalty = min(excess / max_tok, 1.0)
            return 1.0 - penalty

    def __call__(self, prompts, completions, completion_ids=None, **kwargs):
        """
        Compute rewards for generated responses.

        Args:
            prompts: List of input prompts
            completions: List of model outputs
            completion_ids: Ignored
        Returns:
            Tensor of reward scores
        """
        rewards = []

        for i, completion in enumerate(completions):
            # Default mode
            target_mode = "medium"
            for mode in ["low", "medium", "high"]:
                if mode in prompts[i].lower() or mode in completion.lower():
                    target_mode = mode
                    break
            reward = self.compute_mode_reward(completion, target_mode)
            rewards.append(reward)

        return torch.tensor(rewards, dtype=torch.float32)

reward_fn = MultiModeRewardFunction(REASONING_TARGETS)
print("✓ Created multi-mode reward function")
print(f"  Modes: {list(REASONING_TARGETS.keys())}")
for mode, cfg in REASONING_TARGETS.items():
    print(f"  • {mode:8s}: {cfg['min_tokens']:>3d}-{cfg['max_tokens']:>3d} tokens")

## Step 6: Configure and Run GRPO Training with Unsloth

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    optim="adamw_8bit",
    gradient_checkpointing=True,
    num_generations=NUM_GENERATIONS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    report_to="none",
    seed=42,
    dataloader_num_workers=4,
)

print("="*70)
print("GRPO Training Configuration")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Epochs: {NUM_TRAIN_EPOCHS}")
print(f"Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Generations per prompt: {NUM_GENERATIONS}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print("="*70)

# Prepare dataset formatting
def format_prompt(example):
    """Format example for GRPO training."""
    return {
        "prompt": example["prompt"],
        "chosen": example.get("response", ""),
        "mode": example.get("mode", "medium"),
    }

# Format datasets
train_dataset_formatted = train_dataset.map(format_prompt, remove_columns=train_dataset.column_names)
eval_dataset_formatted = eval_dataset.map(format_prompt, remove_columns=eval_dataset.column_names)

print(f"\n✓ Formatted {len(train_dataset_formatted)} training examples")
print(f"✓ Formatted {len(eval_dataset_formatted)} validation examples")
print(f"\nDataset columns: {train_dataset_formatted.column_names}")

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
GRPO Training Configuration
Output directory: saves/llama3-3b/grpo_unsloth_3modes
Epochs: 1.0
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Learning rate: 5e-07
Generations per prompt: 4
Temperature: 0.7
Max new tokens: 1024


Map:   0%|          | 0/27444 [00:00<?, ? examples/s]

Map:   0%|          | 0/1742 [00:00<?, ? examples/s]


✓ Formatted 27444 training examples
✓ Formatted 1742 validation examples

Dataset columns: ['prompt', 'mode', 'chosen']


In [ ]:
print("\nInitializing GRPO Trainer...")

reward_fn.__name__ = "custom_multimode_reward"
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    reward_funcs=[reward_fn],
)


print("✓ GRPO Trainer initialized")
print("\n" + "="*70)
print("Starting GRPO Training with 3 Reasoning Modes!")
print("="*70)
print(f"\nTraining {len(train_dataset_formatted):,} examples")
print(f"  • Low mode:    {len(low_data):,}")
print(f"  • Medium mode: {len(medium_data):,}")
print(f"  • High mode:   {len(high_data):,}")
print(f"\nGenerating {NUM_GENERATIONS} responses per prompt")
print(f"Learning which responses match target reasoning depth\n")
print("="*70)

# Start training
trainer.train()

print("\n" + "="*70)
print("GRPO Training Completed!")
print("="*70)
print(f"Model saved to: {OUTPUT_DIR}")

The model is already on multiple devices. Skipping the move to device specified in `args`.



Initializing GRPO Trainer...
✓ GRPO Trainer initialized

🚀 Starting GRPO Training with 3 Reasoning Modes!

Training 27,444 examples
  • Low mode:    8,630
  • Medium mode: 9,463
  • High mode:   9,351

Generating 4 responses per prompt
Learning which responses match target reasoning depth



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,444 | Num Epochs = 1 | Total steps = 3,430
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / custom_multimode_reward / mean,rewards / custom_multimode_reward / std
10,0.000000,0.239167,0.091009,250.653125,151.200000,256.000000,0.912500,166.975002,125.600000,196.400000,0,0,0,0,0,0.000006,0.239167,0.387061


## Step 7: Save the Trained Model

In [ ]:
print("Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")